# Descriptive Answers

Questions and their resulting answers from our exploration of our data, along with visuals and mathematical evidence.

## Setup

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import os
import glob


# set style for Seaborn
colour_palette = ["#42698E", "#FF8552", "#7C9FC1", "#D6E5F1", "#C0D4E7", "#211A1D"]
sns.set_theme(style="whitegrid", palette=colour_palette, font="Segoe UI", font_scale=1.1)

## Functions

In [2]:
### functions ###
    
def combine_csv_in_folder(data_folder: str, output_file: str="transactions.csv") -> None:
    """
    
    Combine the all the CSVs in data_folder into a single CSV.
    Assuming all the CSVs being combined have the same format.
    
    Adapted from code by: Ishfaq
    
    Input:
        data_folder (str): Name of the folder with all the CSVs.
        output_file (str): Name of the CSV of all the CSVs.
        
    Returns:
        None
        
    """
    # get a list of all CSVs in the data_folder
    all_files = glob.glob(os.path.join(data_folder, "**", "*.csv"), recursive=True)
    print(f"Found {len(all_files)} CSV files.\n")

    # iterate through each CSV, read the CSV, add a column with the demographic group name (folder name)
    dfs = []
    for fp in sorted(all_files):
        print("Loading:", fp)
        df = pd.read_csv(fp)
        folder = os.path.basename(os.path.dirname(fp))   # e.g. cleaned_50up_f_r
        area_type = folder[-1]
        if area_type == "r":
            df["area_type"] = "rural"
        elif area_type == "u":
            df["area_type"] = "urban"
        dfs.append(df)
    
    # combine all the DataFrames into a single one
    data = pd.concat(dfs, ignore_index=True)
    
    # save the combined data DataFrame as a CSV
    data.to_csv(output_file, index=False)
    print(f"CSV files combined and saved to {output_file}.")

## Load Data

In [3]:
# paths
DATA_ROOT = "cleaned_data_files"
combined_data_path = "transactions.csv"

# check if combined data CSV alreday exists and if not make it
if not os.path.isfile(combined_data_path):
    combine_csv_in_folder(DATA_ROOT, combined_data_path)

In [ ]:
# load data
df_raw = pd.read_csv(combined_data_path)

In [ ]:
# view the first few rows
display(df_raw.head())

# get shape of data
print(f"Data Shape: {df_raw.shape}\n")

# check for missing values
print(f"Number of Missing Values per Feature:\n{df_raw.isnull().sum()}\n")

# get column data types
print("Feature Data Info:")
df_raw.info()

# get descriptive stats for numeric columns
print(f"\nDescriptive Statistics for Numerical Columns:")
display(df_raw.describe())

In [ ]:
# check fraud distribution
fraud_counts = df_raw["is_fraud"].value_counts()
print(f"Fraud Transaction Counts: {fraud_counts} \n")

fraud_rate = df_raw['is_fraud'].mean()
print("Fraud Rate:", fraud_rate)

In [ ]:
# filter for cardholders over 18
df = df_raw[df_raw["age"] >= 18]
df.shape

In [ ]:
# check updated farud dsitribution
fraud_counts = df["is_fraud"].value_counts()
print(f"Fraud Transaction Counts: {fraud_counts} \n")

fraud_rate = df['is_fraud'].mean()
print("Fraud Rate:", fraud_rate)

## Descriptive Question 1

**Question**: Do fraudulent transactions have higher amounts?

**Feature**: `amt` (numeric)

**Test**: t-test

In [ ]:
# visual inspection
plt.figure(figsize=(6, 6))

sns.boxplot(x="is_fraud", y="amt", data=df, showfliers=False)
plt.title("Transaction Amount vs Fraud")
plt.xlabel("Fraud (0 = No, 1 = Yes)")
plt.ylabel("Transaction Amount ($)")

plt.show()

In [ ]:
# mathematical interpretation
fraud_amt = df[df["is_fraud"] == 1]["amt"]
nonfraud_amt = df[df["is_fraud"] == 0]["amt"]

# hypotheses
print("**Hypothesis Test for Transaction Amount vs Fraud**\n")
print("Null Hypothesis (H_0): The mean transaction amount is the same for fraudulent and non-fraudulent transactions.")
print("Alternative Hypothesis (H_A): The mean transaction amount for fraudulent transactions is greater than that for non-fraudulent transactions.\n")

# get means
print(f"Mean Amount for Fraud: ${fraud_amt.mean()}")
print(f"Mean Amount for Non-Fraud: ${nonfraud_amt.mean()}")

# two sample t-test assuming unequal variance
t_stat, p_value_two_sided = stats.ttest_ind(fraud_amt, nonfraud_amt, equal_var=False)
p_value_one_sided = p_value_two_sided / 2
print(f"\nt-statistic: {t_stat:.4f}")
print(f"One-sided p-value: {p_value_one_sided:.6f}")

# Cohen's d because large sample size
pooled_sd = np.sqrt(((fraud_amt.var() + nonfraud_amt.var()) / 2))
cohen_d = (fraud_amt.mean() - nonfraud_amt.mean()) / pooled_sd
print(f"Cohen's d: {cohen_d:.4f}")

# conclusion
alpha = 0.01
print(f"Alpha: {alpha}")

if p_value_one_sided < alpha and t_stat > 0:
    print(f"\nSince p-value ({p_value_one_sided:.6f}) < {alpha}, reject the null hypothesis.")
    print("The fraudulent transaction amounts are statistically significantly larger than non-fraudulent transactions.")
else:
    print(f"\nSince p-value ({p_value:.6f}) >= {alpha}, we fail to reject the null hypothesis.")
    
print(f"Cohen's d of {cohen_d:.4f} indicates a large effect size, meaning the difference in transaction amounts is substantial and has practical significance.")

**Answer**: Fraudulent transactions have significantly higher transaction amounts than non-fraudulent transactions. The t-test showed a very large difference in mean transaction amounts (mean for fraud: $535.62 vs non-fraud: $70.35) with a t-statistic of 154.87, a p-value < 0.000001, and a large effect size (Cohen’s d = 1.5447).

## Descriptive Question 2

**Question**: Does fraud vary by category?

**Feature**: `category` (catgeorical)

**Test**: Chi-square

In [ ]:
# get fraud rate per category
category_stats = df.groupby("category")["is_fraud"].agg(["mean", "count"])
category_stats = category_stats.rename(columns={"mean": "fraud_rate","count": "transactions"})

# print stats
print(category_stats.sort_values("fraud_rate", ascending=False))

# reset index and sort for plotting later
plot_category_stats = category_stats.reset_index()
plot_category_stats = plot_category_stats.sort_values("fraud_rate", ascending=False)

In [ ]:
# visual inspection
plt.figure(figsize=(10, 6))
fraud_palette = sns.light_palette("#FF8552", n_colors=len(plot_category_stats), reverse=True)

sns.barplot(x="category", y="fraud_rate", data=plot_category_stats, palette=fraud_palette)
plt.xticks(rotation=45, ha="right")
plt.title("Fraud Rate by Transaction Category")
plt.ylabel("Fraud Rate (%)")
plt.xlabel("Transaction Category")

plt.show()

In [ ]:
# mathematical interpretation
fraud_category_table = pd.crosstab(df["category"], df["is_fraud"])

# hypotheses
print("**Hypothesis Test for Transaction Category vs Fraud**\n")
print("Null Hypothesis (H_0): Fraudulent transactions are independent of transaction category.")
print("Alternative Hypothesis (H_A): Fraudulent transactions depend on transaction category.\n")

print("Fraud Rate by Category:")
print(category_stats)

# chi-sqaure test of independence
chi2_stat, p_value, dof, expected = stats.chi2_contingency(fraud_category_table)
p_value_one_sided = p_value_two_sided / 2
print(f"\nChi-square statistic: {chi2_stat:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6f}")

# Cramer's V because of large sample size
n = fraud_category_table.sum().sum() 
min_dim = min(fraud_category_table.shape) - 1
cramers_v = np.sqrt(chi2_stat / (n * min_dim))
print(f"Cramer's V: {cramers_v:.4f}")

# conclusion
alpha = 0.01
print(f"Alpha: {alpha}")

if p_value < alpha:
    print(f"\nSince p-value ({p_value:.6f}) < {alpha}, reject the null hypothesis.")
    print("There is statistical evidence that fraud transactions depend on transaction category.")
else:
    print(f"\nSince p-value ({p_value:.6f}) >= {alpha}, we fail to reject the null hypothesis.")

print(f"Cramer's V  of {cramers_v:.4f} indicates a weak association between fraud and transaction category, meaning fraud is slightly more likely in certain categories, but the association is not very strong.")

**Answer**: Fraudulent transactions vary significantly across different transaction categories. The chi-square test showed a strong association between fraud and category (p-value < 0.000001), but the effect size (Cramer's V = 0.0716) indicates that while the relationship is statistically significant, it is weak.

## Descriptive Question 3

**Question**: Is age related to fraud?

**Feature**: `age` (numeric)

**Test**: t-test

In [ ]:
# visual inspection
plt.figure(figsize=(6, 6))

sns.boxplot(x="is_fraud", y="age", data=df, showfliers=False)
plt.title("Cardholder Age vs Fraud")
plt.xlabel("Fraud (0 = No, 1 = Yes)")
plt.ylabel("Cardholder Age")

plt.show()

In [ ]:
# mathematical interpretation
fraud_age = df[df["is_fraud"] == 1]["age"]
nonfraud_age = df[df["is_fraud"] == 0]["age"]

# hypotheses
print("**Hypothesis Test for Cradholder Age vs Fraud**\n")
print("Null Hypothesis (H_0): The mean cardholder age is the same for fraudulent and non-fraudulent transactions.")
print("Alternative Hypothesis (H_A): The mean cardholder age for fraudulent transactions is greater than that for non-fraudulent transactions.\n")

# get means
print(f"Mean Amount for Fraud: ${fraud_age.mean()}")
print(f"Mean Amount for Non-Fraud: ${nonfraud_age.mean()}")

# two sample t-test assuming unequal variance
t_stat, p_value_two_sided = stats.ttest_ind(fraud_age, nonfraud_age, equal_var=False)
p_value_one_sided = p_value_two_sided / 2
print(f"\nt-statistic: {t_stat:.4f}")
print(f"One-sided p-value: {p_value_one_sided:.6f}")

# Cohen's d because large sample size
pooled_sd = np.sqrt(((fraud_age.var() + nonfraud_age.var()) / 2))
cohen_d = (fraud_age.mean() - nonfraud_age.mean()) / pooled_sd
print(f"Cohen's d: {cohen_d:.4f}")

# conclusion
alpha = 0.01
print(f"Alpha: {alpha}")

if p_value_one_sided < alpha and t_stat > 0:
    print(f"\nSince p-value ({p_value_one_sided:.6f}) < {alpha}, reject the null hypothesis.")
    print("The mean age for fraudulent transactions is statistically significantly older than that for non-fraudulent transactions.")
else:
    print(f"\nSince p-value ({p_value:.6f}) >= {alpha}, we fail to reject the null hypothesis.")
    
print(f"Cohen's d of {cohen_d:.4f} indicates a small to medium effect. This suggests that fraudulent transactions tend to have slightly older cardholders, but the effect is moderate.")

**Answer**: Fraudulent transactions are more likely to involve cardholders of older ages. The t-test showed a statistically significant difference in the mean age for fraudulent (46.56 years) vs. non-fraudulent transactions (41.43 years), with a t-statistic of 38.92, a p-value < 0.000001, and a moderate effect size (Cohen’s d = 0.3028).

**Note**: Due to the large sample size, the statistical tests have very high power so we get p-values that are too small for Python to represent properly.

## References
- https://seaborn.pydata.org/tutorial/aesthetics.html
- https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html#scipy.stats.ttest_ind
- https://stackoverflow.com/a/52680007